# 06H ENTERPRISE – Automated Model Comparison Dashboard
This notebook automatically loads metrics exported by model notebooks (06A–06G). If metric files are unavailable, it falls back to a template for manual entry.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time
import joblib

metric_files = [
    "logistic_metrics.csv",
    "decision_tree_metrics.csv",
    "random_forest_metrics.csv",
    "xgboost_metrics.csv",
    "lightgbm_metrics.csv",
    "catboost_metrics.csv",
    "svm_metrics.csv"
]

frames=[]
missing=[]

for f in metric_files:
    if Path(f).exists():
        frames.append(pd.read_csv(f))
    else:
        missing.append(f)

if frames:
    results=pd.concat(frames,ignore_index=True)
    print("Loaded metric files:",len(frames))
else:
    print("No metric files found.")
    print("Create these CSV files from notebooks 06A–06G or paste values below.")
    results=pd.DataFrame({
        "Model":["Logistic Regression","Decision Tree","Random Forest","XGBoost","LightGBM","CatBoost","SVM"],
        "Accuracy":[0]*7,
        "Precision":[0]*7,
        "Recall":[0]*7,
        "F1":[0]*7,
        "ROC_AUC":[0]*7
    })

display(results)
if missing:
    print("Missing:",missing)


## Production Ranking

In [ ]:
weights={'Accuracy':0.15,'Precision':0.2,'Recall':0.2,'F1':0.25,'ROC_AUC':0.2}
results['Production_Score']=sum(results[k]*v for k,v in weights.items())
leaderboard=results.sort_values('Production_Score',ascending=False)
display(leaderboard)


## Automatic Inference Benchmark (optional)

In [ ]:
from pathlib import Path
import numpy as np

model_files={
'Logistic Regression':'logistic_regression_model.joblib',
'Decision Tree':'decision_tree_model.joblib',
'Random Forest':'random_forest_model.joblib',
'XGBoost':'xgboost_model.joblib',
'LightGBM':'lightgbm_model.joblib',
'CatBoost':'catboost_model.joblib',
'SVM':'svm_model.joblib'
}

if Path("american_bankruptcy.csv").exists():
    df=pd.read_csv("american_bankruptcy.csv")
    drop=[c for c in ["status_label","target","company_name"] if c in df.columns]
    X=df.drop(columns=drop).head(100)
    timings=[]
    for name,file in model_files.items():
        if Path(file).exists():
            model=joblib.load(file)
            t=time.perf_counter()
            _=model.predict(X)
            timings.append([name,(time.perf_counter()-t)*1000])
    if timings:
        bench=pd.DataFrame(timings,columns=["Model","Inference_ms_100rows"])
        display(bench)


## Visual Dashboard

In [ ]:
for metric in ['Accuracy','Precision','Recall','F1','ROC_AUC']:
    plt.figure(figsize=(8,4))
    plt.bar(leaderboard['Model'],leaderboard[metric])
    plt.xticks(rotation=30,ha='right')
    plt.title(metric)
    plt.tight_layout()
    plt.show()


## Executive Report

In [ ]:
winner=leaderboard.iloc[0]
report=pd.DataFrame({
'Item':['Recommended Model','Production Score','Reason'],
'Value':[winner['Model'],round(winner['Production_Score'],4),
'Highest weighted score across evaluation metrics']
})
display(report)

leaderboard.to_csv('enterprise_model_leaderboard.csv',index=False)
print('Saved enterprise_model_leaderboard.csv')
